# Team Fit Recommender (Wingers)

Combine role-fit scores into team-specific recruitment rankings.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/processed")
ROLE_SCORES_PATH = DATA_DIR / "winger_role_scores.parquet"
OUT_PATH = DATA_DIR / "team_fit_shortlists.csv"

TOP_N = 30
MIN_MINUTES = 300
MIN_STABILITY = 0.60


In [2]:
df = pd.read_parquet(ROLE_SCORES_PATH)
print("Input shape:", df.shape)

required_cols = [
    "score_direct_runner_adj",
    "score_playmaker_winger_adj",
    "score_inside_forward_adj",
    "minutes",
    "cluster_agreement",
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[(df["minutes"] >= MIN_MINUTES) & (df["cluster_agreement"] >= MIN_STABILITY)].copy()
print("Eligible shape:", df.shape)


Input shape: (1438, 22)
Eligible shape: (3, 22)


In [3]:
# Team style presets (weights must sum to 1.0)
TEAM_STYLES = {
    "transition_heavy": {
        "direct_runner": 0.55,
        "playmaker_winger": 0.15,
        "inside_forward": 0.30,
    },
    "possession_heavy": {
        "direct_runner": 0.20,
        "playmaker_winger": 0.55,
        "inside_forward": 0.25,
    },
    "balanced_attack": {
        "direct_runner": 0.34,
        "playmaker_winger": 0.33,
        "inside_forward": 0.33,
    },
}

for style, w in TEAM_STYLES.items():
    total = sum(w.values())
    if abs(total - 1.0) > 1e-9:
        raise ValueError(f"Weights for {style} sum to {total}, expected 1.0")

TEAM_STYLES


{'transition_heavy': {'direct_runner': 0.55,
  'playmaker_winger': 0.15,
  'inside_forward': 0.3},
 'possession_heavy': {'direct_runner': 0.2,
  'playmaker_winger': 0.55,
  'inside_forward': 0.25},
 'balanced_attack': {'direct_runner': 0.34,
  'playmaker_winger': 0.33,
  'inside_forward': 0.33}}

In [4]:
def rank_for_style(df_in: pd.DataFrame, style_name: str, weights: dict, top_n: int = 30) -> pd.DataFrame:
    out = df_in.copy()
    out["team_style"] = style_name

    out["team_fit_score"] = (
        weights["direct_runner"] * out["score_direct_runner_adj"] +
        weights["playmaker_winger"] * out["score_playmaker_winger_adj"] +
        weights["inside_forward"] * out["score_inside_forward_adj"]
    )

    cols = [
        "team_style",
        "player_id",
        "player_name" if "player_name" in out.columns else "player_id",
        "cluster",
        "cluster_name",
        "minutes",
        "cluster_agreement",
        "score_direct_runner_adj",
        "score_playmaker_winger_adj",
        "score_inside_forward_adj",
        "team_fit_score",
    ]

    return (
        out.sort_values("team_fit_score", ascending=False)[cols]
        .head(top_n)
        .reset_index(drop=True)
    )


In [5]:
tables = []
for style_name, weights in TEAM_STYLES.items():
    table = rank_for_style(df, style_name, weights, top_n=TOP_N)
    tables.append(table)

team_fit_shortlists = pd.concat(tables, ignore_index=True)
team_fit_shortlists.head(20)


,team_style,player_id,player_name,cluster,cluster_name,minutes,cluster_agreement,score_direct_runner_adj,score_playmaker_winger_adj,score_inside_forward_adj,team_fit_score
0,transition_heavy,6907,Pedro León Sánchez Gil,0,Inside Forward Finisher,4420.0,0.6,41.265647,39.179416,37.923505,39.950070
1,transition_heavy,3083,Heung-Min Son,0,Inside Forward Finisher,3250.0,0.6,38.497218,33.577886,44.689152,39.616898
2,transition_heavy,8233,Gonzalo Castro Randon,0,Inside Forward Finisher,1560.0,0.6,37.735744,44.397079,27.014604,35.518602
3,possession_heavy,6907,Pedro León Sánchez Gil,0,Inside Forward Finisher,4420.0,0.6,41.265647,39.179416,37.923505,39.282684
4,possession_heavy,8233,Gonzalo Castro Randon,0,Inside Forward Finisher,1560.0,0.6,37.735744,44.397079,27.014604,38.719193
5,possession_heavy,3083,Heung-Min Son,0,Inside Forward Finisher,3250.0,0.6,38.497218,33.577886,44.689152,37.339569
6,balanced_attack,6907,Pedro León Sánchez Gil,0,Inside Forward Finisher,4420.0,0.6,41.265647,39.179416,37.923505,39.474284
7,balanced_attack,3083,Heung-Min Son,0,Inside Forward Finisher,3250.0,0.6,38.497218,33.577886,44.689152,38.917177
8,balanced_attack,8233,Gonzalo Castro Randon,0,Inside Forward Finisher,1560.0,0.6,37.735744,44.397079,27.014604,36.396008


In [6]:
team_fit_shortlists.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
print("Rows:", len(team_fit_shortlists))

for style in TEAM_STYLES.keys():
    print("\nTop 5 for", style)
    display(team_fit_shortlists[team_fit_shortlists["team_style"] == style].head(5))


Saved: ..\data\processed\team_fit_shortlists.csv
Rows: 9

Top 5 for transition_heavy


,team_style,player_id,player_name,cluster,cluster_name,minutes,cluster_agreement,score_direct_runner_adj,score_playmaker_winger_adj,score_inside_forward_adj,team_fit_score
0,transition_heavy,6907,Pedro León Sánchez Gil,0,Inside Forward Finisher,4420.0,0.6,41.265647,39.179416,37.923505,39.950070
1,transition_heavy,3083,Heung-Min Son,0,Inside Forward Finisher,3250.0,0.6,38.497218,33.577886,44.689152,39.616898
2,transition_heavy,8233,Gonzalo Castro Randon,0,Inside Forward Finisher,1560.0,0.6,37.735744,44.397079,27.014604,35.518602



Top 5 for possession_heavy


,team_style,player_id,player_name,cluster,cluster_name,minutes,cluster_agreement,score_direct_runner_adj,score_playmaker_winger_adj,score_inside_forward_adj,team_fit_score
3,possession_heavy,6907,Pedro León Sánchez Gil,0,Inside Forward Finisher,4420.0,0.6,41.265647,39.179416,37.923505,39.282684
4,possession_heavy,8233,Gonzalo Castro Randon,0,Inside Forward Finisher,1560.0,0.6,37.735744,44.397079,27.014604,38.719193
5,possession_heavy,3083,Heung-Min Son,0,Inside Forward Finisher,3250.0,0.6,38.497218,33.577886,44.689152,37.339569



Top 5 for balanced_attack


,team_style,player_id,player_name,cluster,cluster_name,minutes,cluster_agreement,score_direct_runner_adj,score_playmaker_winger_adj,score_inside_forward_adj,team_fit_score
6,balanced_attack,6907,Pedro León Sánchez Gil,0,Inside Forward Finisher,4420.0,0.6,41.265647,39.179416,37.923505,39.474284
7,balanced_attack,3083,Heung-Min Son,0,Inside Forward Finisher,3250.0,0.6,38.497218,33.577886,44.689152,38.917177
8,balanced_attack,8233,Gonzalo Castro Randon,0,Inside Forward Finisher,1560.0,0.6,37.735744,44.397079,27.014604,36.396008
